In [ ]:
%load_ext autoreload
%autoreload 2

import warnings

import matplotlib.pyplot as plt
import numpy as np
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)

from vetuner.correction import select_samples
from vetuner.drive_cycle import VALIDATION_CYCLE, generate_drive_cycle
from vetuner.engine_model import simulate_engine
from vetuner.gating import gate_samples
from vetuner.iteration import iterate_calibration
from vetuner.metrics import evaluate_afr, evaluate_table
from vetuner.pulse_width import pulse_width_map
from vetuner.sensors import apply_sensor_model
from vetuner.surface_fit import cell_sample_counts
from vetuner.ve_surface import (
    BASE_MAP_SURFACE,
    DEFAULT_MAP_AXIS,
    DEFAULT_RPM_AXIS,
    TRUE_SURFACE,
    ve_table,
)

train = generate_drive_cycle()
test = generate_drive_cycle(cycle=VALIDATION_CYCLE)
base = ve_table(params=BASE_MAP_SURFACE)
truth = ve_table(params=TRUE_SURFACE)

sim = simulate_engine(train, base)
log = apply_sensor_model(sim, seed=0)
kept = select_samples(log, gate_samples(log).accepted)
counts = cell_sample_counts(kept.rpm, kept.map_kpa)

history = iterate_calibration(train, base, gain=0.6, max_passes=5, method="gaussian_process")
calibrated = history.final_table
result = pulse_width_map(calibrated)

extent = [DEFAULT_RPM_AXIS[0], DEFAULT_RPM_AXIS[-1],
          DEFAULT_MAP_AXIS[0], DEFAULT_MAP_AXIS[-1]]

print(f"{len(log)} samples, {len(kept)} accepted, "
      f"{(counts > 0).sum()} of {counts.size} cells reached")
print(f"table RMSE {evaluate_table(calibrated, truth).rmse:.2f} pts, "
      f"held-out AFR error {evaluate_afr(calibrated, test).mean_abs_pct:.2f}%")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from vetuner.drive_cycle import VALIDATION_CYCLE, generate_drive_cycle
from vetuner.engine_model import simulate_engine
from vetuner.metrics import evaluate_afr

test = generate_drive_cycle(cycle=VALIDATION_CYCLE)

before = simulate_engine(test, base)
after = simulate_engine(test, calibrated)

fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True,
                         gridspec_kw={"height_ratios": [1, 2, 2]})

axes[0].plot(test.time_s, test.rpm, lw=1.0, color="tab:blue")
axes[0].set_ylabel("rpm")
axes[0].set_title("Held-out validation cycle")

for ax, sim, label in [(axes[1], before, "starting map"), (axes[2], after, "calibrated")]:
    ax.plot(sim.time_s, sim.afr_target, lw=1.2, color="black", ls="--", label="target")
    ax.plot(sim.time_s, sim.afr_actual, lw=1.0, color="tab:red", label="actual")
    ax.fill_between(sim.time_s, sim.afr_target, sim.afr_actual,
                    color="tab:red", alpha=0.15)
    error = evaluate_afr(base if label == "starting map" else calibrated, test)
    ax.set_ylabel("AFR")
    ax.set_title(f"{label} — mean error {error.mean_abs_pct:.2f}%")
    ax.legend(loc="upper right", fontsize=8)
    ax.set_ylim(11, 16)

axes[2].set_xlabel("time (s)")
for ax in axes:
    ax.grid(alpha=0.3)

fig.tight_layout()

In [ ]:
from vetuner.surface_fit import cell_sample_counts
from vetuner.ve_surface import DEFAULT_MAP_AXIS, DEFAULT_RPM_AXIS

extent = [DEFAULT_RPM_AXIS[0], DEFAULT_RPM_AXIS[-1],
          DEFAULT_MAP_AXIS[0], DEFAULT_MAP_AXIS[-1]]

fig, axes = plt.subplots(1, 4, figsize=(18, 4))

# Input: where the log went
axes[0].scatter(log.rpm, log.map_kpa, s=2, alpha=0.2, color="grey", label="all samples")
axes[0].scatter(kept.rpm, kept.map_kpa, s=2, alpha=0.4, color="tab:green", label="accepted")
axes[0].set_title(f"INPUT: log coverage\n{len(log)} samples, {len(kept)} accepted")
axes[0].set_xlabel("rpm")
axes[0].set_ylabel("MAP (kPa abs)")
axes[0].legend(fontsize=8, markerscale=4)

# Input: data density per cell
im = axes[1].imshow(counts.T, origin="lower", aspect="auto", extent=extent, cmap="magma")
axes[1].set_title(f"INPUT: samples per cell\n{(counts > 0).sum()} of {counts.size} cells reached")
axes[1].set_xlabel("rpm")
fig.colorbar(im, ax=axes[1])

# Output: the table
im = axes[2].imshow(calibrated.T * 100, origin="lower", aspect="auto",
                    extent=extent, cmap="viridis")
axes[2].set_title("OUTPUT: corrected VE table (%)")
axes[2].set_xlabel("rpm")
fig.colorbar(im, ax=axes[2])

# Output: the injector command
im = axes[3].imshow(result.pulse_width_ms.T, origin="lower", aspect="auto",
                    extent=extent, cmap="cividis")
axes[3].set_title(f"OUTPUT: pulse width (ms)\npeak duty {result.max_duty:.0%}")
axes[3].set_xlabel("rpm")
fig.colorbar(im, ax=axes[3])

fig.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

error_before = (base - truth) * 100
error_after = (calibrated - truth) * 100
limit = np.abs(error_before).max()

for ax, data, title in [
    (axes[0], error_before, f"Before: RMSE {np.sqrt((error_before**2).mean()):.2f} pts"),
    (axes[1], error_after, f"After: RMSE {np.sqrt((error_after**2).mean()):.2f} pts"),
]:
    im = ax.imshow(data.T, origin="lower", aspect="auto", extent=extent,
                   cmap="RdBu_r", vmin=-limit, vmax=limit)
    ax.set_title(title)
    ax.set_xlabel("rpm")
    fig.colorbar(im, ax=ax)

axes[2].hist(error_before.ravel(), bins=30, alpha=0.6, label="before", color="tab:red")
axes[2].hist(error_after.ravel(), bins=30, alpha=0.6, label="after", color="tab:green")
axes[2].axvline(0, color="black", lw=1)
axes[2].set_xlabel("table error (VE points)")
axes[2].set_ylabel("cells")
axes[2].set_title("Error distribution")
axes[2].legend()

axes[0].set_ylabel("MAP (kPa abs)")
fig.tight_layout()